
# Step 2 Internal SMC Genomics Analysis Notebook



In [ ]:

# Core imports
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown
import geopandas as gpd

import warnings
warnings.filterwarnings("ignore")

# Global style for publication-quality figures
plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.titleweight": "bold",
    "axes.labelsize": 12,
    "axes.labelweight": "bold",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

BASE = Path('/mnt/hpc_acegid/home/khadmig/work/data/malaria/smc-impact-under5-malaria-africa/results').resolve()
FIG_DIR = BASE / 'figures'
FIG_DIR.mkdir(exist_ok=True, parents=True)

Base_res = BASE / "genomics_internal_smc_2021" 
Base_step3 = BASE / "genomics_internal_smc_2021_step3"
manifest_step3 = Base_step3 / "step3_pipeline_manifest.json"

MANIFEST_PATH = Base_res / 'step2_pipeline_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text())
manifest


In [ ]:

# File loading helpers

def local_path_from_manifest(key: str) -> Path:
    return Base_res / Path(manifest['outputs'][key]).name

sample_manifest = pd.read_csv(local_path_from_manifest('sample_manifest_tsv'), sep='	')
frag_cov = pd.read_csv(local_path_from_manifest('fragment_coverage_by_sample_tsv'), sep='	')
frag_cov_state = pd.read_csv(local_path_from_manifest('fragment_coverage_summary_by_state_tsv'), sep='	')
frag_cov_sample = pd.read_csv(local_path_from_manifest('fragment_coverage_summary_by_sample_tsv'), sep='	')
marker_calls = pd.read_csv(local_path_from_manifest('marker_calls_by_sample_long_tsv'), sep='	')
marker_prev_state = pd.read_csv(local_path_from_manifest('marker_prevalence_by_state_tsv'), sep='	')
marker_prev_overall = pd.read_csv(local_path_from_manifest('marker_prevalence_overall_tsv'), sep='	')
marker_matrix = pd.read_csv(local_path_from_manifest('marker_matrix_by_sample_tsv'), sep='	')
marker_heatmap_state = pd.read_csv(local_path_from_manifest('marker_heatmap_by_state_tsv'), sep='	')
marker_callability = pd.read_csv(local_path_from_manifest('marker_callability_by_state_tsv'), sep='	')
hap_calls = pd.read_csv(local_path_from_manifest('haplotype_calls_by_sample_long_tsv'), sep='	')
hap_prev_state = pd.read_csv(local_path_from_manifest('haplotype_prevalence_by_state_tsv'), sep='	')
hap_prev_overall = pd.read_csv(local_path_from_manifest('haplotype_prevalence_overall_tsv'), sep='	')
hap_matrix = pd.read_csv(local_path_from_manifest('haplotype_matrix_by_sample_tsv'), sep='	')
hap_heatmap_state = pd.read_csv(local_path_from_manifest('haplotype_heatmap_by_state_tsv'), sep='	')
expl_state = pd.read_csv(local_path_from_manifest('exploratory_observed_mutation_prevalence_by_state_tsv'), sep='	')
expl_overall = pd.read_csv(local_path_from_manifest('exploratory_observed_mutation_prevalence_overall_tsv'), sep='	')
qc_state = pd.read_csv(local_path_from_manifest('qc_summary_by_state_tsv'), sep='	')
inclusion = pd.read_csv(local_path_from_manifest('inclusion_flow_by_state_tsv'), sep='	')

for df in [frag_cov, frag_cov_state, frag_cov_sample, marker_calls, marker_prev_state, marker_prev_overall,
           marker_matrix, marker_callability, hap_calls, hap_prev_state, hap_prev_overall, hap_matrix,
           qc_state, inclusion]:
    for c in df.columns:
        if c in {'year', 'n_reads', 'mean_reads', 'median_reads', 'min_reads', 'max_reads', 'numerator', 'denominator',
                 'prevalence', 'callable_fraction', 'n_samples', 'n_callable', 'n_present', 'n_fragments',
                 'n_callable_fragments', 'n_selected_samples', 'n_with_bam', 'n_with_vcf',
                 'n_with_any_callable_fragment', 'n_with_any_callable_marker', 'n_with_any_callable_haplotype'}:
            df[c] = pd.to_numeric(df[c], errors='coerce')

state_order = sorted(sample_manifest['state'].dropna().astype(str).unique())
sp_markers = ['N51I', 'C59R', 'S108N', 'I164L', 'A437G', 'K540E', 'A581G', 'A613S']
aq_markers = ['N86Y', 'Y184F', 'D1246Y', 'M74I', 'N75E', 'K76T']
sp_haps = ['dhfr_triple', 'sp_quadruple', 'sp_quintuple', 'sp_sextuple', 'sp_sextuple_dhfr164']
aq_haps = ['mdr1_86Y', 'mdr1_184F', 'mdr1_double', 'pfcrt_76T', 'pfcrt_triple', 'aq_resistant_profile']
haplotype_mutations = {
    'dhfr_triple': 'N51I + C59R + S108N',
    'sp_quadruple': 'A437G + N51I + C59R + S108N',
    'sp_quintuple': 'A437G + K540E + N51I + C59R + S108N',
    'sp_sextuple': 'A437G + K540E + A581G + N51I + C59R + S108N',
    'sp_sextuple_dhfr164': 'A437G + K540E + N51I + C59R + S108N + I164L',
    'mdr1_86Y': 'N86Y',
    'mdr1_184F': 'Y184F',
    'mdr1_double': 'N86Y + Y184F',
    'pfcrt_76T': 'K76T',
    'pfcrt_triple': 'M74I + N75E + K76T',
    'aq_resistant_profile': 'N86Y + Y184F + K76T',
}

def save_figure(fig, name):
    png = FIG_DIR / f'{name}.png'
    pdf = FIG_DIR / f'{name}.pdf'
    tif = FIG_DIR / f'{name}.tif'
    fig.savefig(png, dpi=600 ,  bbox_inches='tight')
    fig.savefig(pdf, dpi=600 ,  bbox_inches='tight')
    fig.savefig(tif, dpi=600 ,  bbox_inches='tight')

    print (png)

    return png, pdf

print('Loaded tables successfully.')
print('Figure directory:', FIG_DIR)


In [ ]:

# Quick data audit
summary = pd.DataFrame({
    'table': [
        'sample_manifest', 'fragment_coverage_by_sample', 'fragment_coverage_summary_by_state',
        'fragment_coverage_summary_by_sample', 'marker_calls', 'marker_prevalence_by_state',
        'marker_prevalence_overall', 'haplotype_calls', 'haplotype_prevalence_by_state',
        'haplotype_prevalence_overall', 'exploratory_by_state', 'exploratory_overall'
    ],
    'rows': [
        len(sample_manifest), len(frag_cov), len(frag_cov_state), len(frag_cov_sample), len(marker_calls),
        len(marker_prev_state), len(marker_prev_overall), len(hap_calls), len(hap_prev_state), len(hap_prev_overall),
        len(expl_state), len(expl_overall)
    ]
})
display(summary)


## 1. Cohort construction and quality control

In [ ]:

# Figure 1. Overall cohort waterfall
fig, ax = plt.subplots(figsize=(10, 5))
steps = ['Selected', 'BAM', 'VCF', 'Callable fragment', 'Callable marker', 'Callable haplotype']
vals = [
    int(inclusion['n_selected_samples'].sum()),
    int(inclusion['n_with_bam'].sum()),
    int(inclusion['n_with_vcf'].sum()),
    int(inclusion['n_with_any_callable_fragment'].sum()),
    int(inclusion['n_with_any_callable_marker'].sum()),
    int(inclusion['n_with_any_callable_haplotype'].sum()),
]
colors = ['#1f3b73', '#2c7fb8', '#41b6c4', '#7fcdbb', '#c7e9b4', '#fdd49e']
ax.bar(range(len(steps)), vals, color=colors, edgecolor='black', linewidth=1.0)
for i, v in enumerate(vals):
    ax.text(i, v + max(vals) * 0.01, f'{v:,}', ha='center', va='bottom', fontweight='bold')
ax.set_xticks(range(len(steps)))
ax.set_xticklabels(steps, rotation=20, ha='right', fontweight='bold')
ax.set_ylabel('Number of samples', fontweight='bold')
ax.set_title('Overall analytical inclusion cascade', fontweight='bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
#save_figure(fig, 'figure_step2_01_overall_inclusion_waterfall')
#plt.show()


In [ ]:

# Figure 2. State-level inclusion summary
plot_df = inclusion.copy().sort_values('n_selected_samples', ascending=False)
metrics = [
    'n_selected_samples', 'n_with_bam', 'n_with_vcf',
    'n_with_any_callable_fragment', 'n_with_any_callable_marker', 'n_with_any_callable_haplotype'
]
labels = ['Selected', 'BAM', 'VCF', 'Callable fragment', 'Callable marker', 'Callable haplotype']
colors = ['#08306b', '#08519c', '#2171b5', '#4292c6', '#6baed6', '#9ecae1']

fig, ax = plt.subplots(figsize=(16, 8))
x = np.arange(len(plot_df))
width = 0.13
for i, (m, lab, col) in enumerate(zip(metrics, labels, colors)):
    ax.bar(x + (i - 2.5) * width, plot_df[m], width=width, label=lab, color=col, edgecolor='black', linewidth=0.6)
ax.set_xticks(x)
ax.set_xticklabels(plot_df['state'], rotation=60, ha='right', fontweight='bold')
ax.set_ylabel('Number of samples', fontweight='bold')
ax.set_title('State-level analytical retention', fontweight='bold')
leg = ax.legend(frameon=False, ncol=3)
for t in leg.get_texts():
    t.set_fontweight('bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')
#save_figure(fig, 'figure_step2_02_state_level_inclusion_bars')
plt.show()


In [ ]:
count_df = (
    sample_manifest.groupby("state", as_index=False)
    .agg(n_samples=("sample_id", "nunique"))
)

nga = gpd.read_file("../assets/gadm41_NGA_1.shp").copy()

state_col = None
for c in ["NAME_1", "name_1", "NAME1", "state", "State"]:
    if c in nga.columns:
        state_col = c
        break

nga["state"] = nga[state_col].astype(str).str.strip()

state_fix_manifest = {
    "Federal Capital Territory": "Abuja Federal Capital Territory",
    "FCT": "Abuja Federal Capital Territory",
}
state_fix_map = {
    "Abuja Federal Capital Territory": "Abuja Federal Capital Territory",
}

count_df["state"] = count_df["state"].replace(state_fix_manifest)
nga["state"] = nga["state"].replace(state_fix_map)

map_df = nga.merge(count_df, on="state", how="left")
map_df["n_samples"] = pd.to_numeric(map_df["n_samples"], errors="coerce")
map_df.loc[map_df["n_samples"] == 0, "n_samples"] = pd.NA

fig, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(18, 10),
    dpi=600,
    gridspec_kw={"width_ratios": [1.0, 1.15]}
)

ax1, ax2 = axes

bar_df = count_df.sort_values("n_samples", ascending=True).copy()
bars = ax1.barh(
    bar_df["state"],
    bar_df["n_samples"],
    color="#225ea8",
    edgecolor="black",
    linewidth=1.0
)

offset = max(1, bar_df["n_samples"].max() * 0.01)
for bar, v in zip(bars, bar_df["n_samples"]):
    ax1.text(
        v + offset,
        bar.get_y() + bar.get_height() / 2,
        f"{int(v)}",
        va="center",
        ha="left",
        fontsize=10,
        fontweight="bold"
    )

ax1.set_xlabel("Number of samples", fontweight="bold", fontsize=14)
ax1.set_ylabel("State", fontweight="bold", fontsize=14)
ax1.set_title("A", loc="left", fontsize=18, fontweight="bold", y=1.02)
ax1.tick_params(axis="x", labelsize=11, width=1.2)
ax1.tick_params(axis="y", labelsize=11, width=1.2)
for label in ax1.get_xticklabels() + ax1.get_yticklabels():
    label.set_fontweight("bold")
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["left"].set_linewidth(1.2)
ax1.spines["bottom"].set_linewidth(1.2)

ax2.set_facecolor("#cfefff")

map_df.plot(
    column="n_samples",
    cmap="YlOrRd",
    linewidth=0.9,
    edgecolor="black",
    legend=True,
    legend_kwds={"label": "Number of samples", "shrink": 0.78},
    missing_kwds={"color": "white"},
    ax=ax2
)

for _, row in map_df.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    if pd.isna(row["n_samples"]):
        continue
    pt = row.geometry.representative_point()
    ax2.text(
        pt.x,
        pt.y,
        f"{row['state']}\n{int(row['n_samples'])}",
        ha="center",
        va="center",
        fontsize=7.2,
        fontweight="bold",
        color="black",
        bbox=dict(
            facecolor="white",
            edgecolor="black",
            boxstyle="round,pad=0.18",
            alpha=0.78,
            linewidth=0.6
        ),
        zorder=5
    )

ax2.set_title("B", loc="left", fontsize=18, fontweight="bold", y=1.35)
ax2.set_axis_off()

if len(fig.axes) > 2:
    cax = fig.axes[-1]
    cax.set_ylabel("Number of samples", fontweight="bold", fontsize=13)
    cax.tick_params(labelsize=10, width=1.0)
    for lab in cax.get_yticklabels():
        lab.set_fontweight("bold")

fig.suptitle("Internal 2021 cohort size by state\n\n", fontsize=20, fontweight="bold", y=0.97)
fig.subplots_adjust(top=0.90, wspace=0.06)
save_figure(fig, "figure_step2_03_sample_counts_by_state_and_nigeria_map")
plt.show()

## 2. Coverage and fragment performance

In [ ]:

# Figure 4. Callable fraction heatmap by fragment and state
call_frac = frag_cov_state.pivot_table(index='state', columns='reference_id', values='callable_fraction', aggfunc='first')
call_frac = call_frac.reindex(index=state_order)
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(call_frac.values, aspect='auto', vmin=0, vmax=1, cmap='viridis')
ax.set_xticks(range(call_frac.shape[1]))
ax.set_xticklabels(call_frac.columns, rotation=45, ha='right', fontweight='bold')
ax.set_yticks(range(call_frac.shape[0]))
ax.set_yticklabels(call_frac.index, fontweight='bold')
ax.set_title('Callable fraction by fragment and state', fontweight='bold')
ax.set_xlabel('Fragment / target gene', fontweight='bold')
ax.set_ylabel('State', fontweight='bold')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Callable fraction', fontweight='bold')
for t in cbar.ax.get_yticklabels():
    t.set_fontweight('bold')
#save_figure(fig, 'figure_step2_04_callable_fraction_heatmap_state_fragment')
#plt.show()


In [ ]:

# Figure 5. Median reads heatmap by fragment and state
med_cov = frag_cov_state.pivot_table(index='state', columns='reference_id', values='median_reads', aggfunc='first')
med_cov = med_cov.reindex(index=state_order)
fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(med_cov.values, aspect='auto', cmap='magma')
ax.set_xticks(range(med_cov.shape[1]))
ax.set_xticklabels(med_cov.columns, rotation=45, ha='right', fontweight='bold')
ax.set_yticks(range(med_cov.shape[0]))
ax.set_yticklabels(med_cov.index, fontweight='bold')
ax.set_title('Median read depth by fragment and state', fontweight='bold')
ax.set_xlabel('Fragment / target gene', fontweight='bold')
ax.set_ylabel('State', fontweight='bold')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Median reads', fontweight='bold')
for t in cbar.ax.get_yticklabels():
    t.set_fontweight('bold')
#save_figure(fig, 'figure_step2_05_median_read_depth_heatmap_state_fragment')
#plt.show()


In [ ]:

# Figure 6. Sample-level callable fragment burden
fig, ax = plt.subplots(figsize=(8, 5))
vals = frag_cov_sample['n_callable_fragments'].dropna().astype(int)
bins = np.arange(vals.min(), vals.max() + 2) - 0.5
ax.hist(vals, bins=bins, edgecolor='black', color='#238b45')
ax.set_xticks(sorted(vals.unique()))
ax.set_xlabel('Number of callable fragments per sample', fontweight='bold')
ax.set_ylabel('Number of samples', fontweight='bold')
ax.set_title('Distribution of callable fragments per sample', fontweight='bold')
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')
#save_figure(fig, 'figure_step2_06_callable_fragment_distribution_per_sample')
#plt.show()


## 3. Resistance marker prevalence

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plot_df = marker_prev_overall.copy().sort_values(
    ['drug', 'prevalence', 'mutation'],
    ascending=[True, False, True]
)

fig, ax = plt.subplots(figsize=(11, 6))

colors = ['#b2182b' if d == 'SP' else '#2166ac' for d in plot_df['drug']]
bars = ax.bar(
    plot_df['mutation'],
    plot_df['prevalence'],
    color=colors,
    edgecolor='black',
    linewidth=1.0
)

for i, row in plot_df.reset_index(drop=True).iterrows():
    ax.text(
        i,
        row['prevalence'] + 0.02,
        f"{int(row['numerator'])}/{int(row['denominator'])}",
        ha='center',
        va='bottom',
        rotation=90,
        fontsize=9,
        fontweight='bold'
    )

ax.set_ylim(0, 1.1)
ax.set_ylabel('Prevalence', fontweight='bold', fontsize=13)
ax.set_xlabel('Mutation', fontweight='bold', fontsize=13)
ax.set_title('Overall prevalence of validated resistance markers', fontweight='bold', fontsize=16)

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df['mutation'], rotation=45, ha='right', fontweight='bold')

for label in ax.get_yticklabels():
    label.set_fontweight('bold')

legend_elements = [
    Patch(facecolor='#b2182b', edgecolor='black', label='SP markers (Sulfadoxine–Pyrimethamine)'),
    Patch(facecolor='#2166ac', edgecolor='black', label='AQ markers (Amodiaquine)'),
    Line2D([0], [0], color='black', lw=0, label='Bars = mutation prevalence'),
    Line2D([0], [0], color='black', lw=0, label='Text = number of samples with mutation / callable samples')
]

leg = ax.legend(
    handles=legend_elements,
    frameon=True,
    fontsize=7,
    loc='upper right',
    title='Legend',
    title_fontsize=11
)

leg.get_title().set_fontweight('bold')
for t in leg.get_texts():
    t.set_fontweight('bold')

ax.grid(axis='y', linestyle='--', alpha=0.3)

fig.tight_layout()
save_figure(fig, 'figure_step2_07_overall_marker_prevalence')
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np

sp_df = marker_prev_state[marker_prev_state["mutation"].isin(sp_markers)] \
    .pivot_table(index="state", columns="mutation", values="prevalence", aggfunc="first")

sp_df = sp_df.reindex(index=state_order, columns=sp_markers)

nga = gpd.read_file("../assets/gadm41_NGA_1.shp").copy()

state_col = None
for c in ["NAME_1", "name_1", "NAME1", "state", "State"]:
    if c in nga.columns:
        state_col = c
        break

nga["state"] = nga[state_col].astype(str).str.strip()

state_fix_manifest = {
    "Federal Capital Territory": "Abuja Federal Capital Territory",
    "FCT": "Abuja Federal Capital Territory",
}
state_fix_map = {
    "Abuja Federal Capital Territory": "Abuja Federal Capital Territory",
}

sp_df = sp_df.rename(index=state_fix_manifest)
nga["state"] = nga["state"].replace(state_fix_map)

n_mut = len(sp_markers)
ncols = 4
nrows = int(np.ceil(n_mut / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(22, 10 * nrows / 2), dpi=600)
axes = axes.flatten()

letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("#f7f7f7")

for i, mut in enumerate(sp_markers):
    ax = axes[i]

    tmp = sp_df[[mut]].reset_index().rename(columns={mut: "prevalence"})
    tmp["label_value"] = tmp["prevalence"]
    tmp.loc[tmp["label_value"] == 0, "label_value"] = np.nan

    map_df = nga.merge(tmp, on="state", how="left")
    map_df["plot_value"] = map_df["prevalence"]
    map_df.loc[map_df["plot_value"] == 0, "plot_value"] = 0.001

    map_df.plot(
        column="plot_value",
        cmap=cmap,
        vmin=0,
        vmax=1,
        linewidth=0.8,
        edgecolor="black",
        ax=ax,
        legend=False,
        missing_kwds={"color": "#f7f7f7", "edgecolor": "black", "linewidth": 0.8}
    )

    ax.set_title(f"{letters[i]})  {mut}", loc="left", fontsize=16, fontweight="bold")
    ax.set_axis_off()

    for _, row in map_df.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        if pd.isna(row["prevalence"]):
            continue

        pt = row.geometry.representative_point()

        if pd.isna(row["label_value"]):
            label_text = f"{row['state']}\n0.00"
            bbox_fc = "#fff2f0"
            txt_color = "#b2182b"
        else:
            label_text = f"{row['state']}\n{row['label_value']:.2f}"
            bbox_fc = "white"
            txt_color = "black"

        ax.text(
            pt.x,
            pt.y,
            label_text,
            ha="center",
            va="center",
            fontsize=6.2,
            fontweight="bold",
            color=txt_color,
            bbox=dict(
                facecolor=bbox_fc,
                alpha=0.82,
                boxstyle="round,pad=0.16",
                linewidth=0.35,
                edgecolor="black"
            ),
            zorder=5
        )

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

cax = fig.add_axes([0.92, 0.25, 0.015, 0.5])
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Prevalence", fontweight="bold", fontsize=13)
for t in cbar.ax.get_yticklabels():
    t.set_fontweight("bold")

fig.suptitle("SP marker prevalence by state", fontsize=22, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 0.96])

save_figure(fig, "figure_step2_sp_marker_maps_multi_panel")
plt.show()

In [ ]:
aq_present = [m for m in aq_markers if m in marker_prev_state["mutation"].unique()]

aq_df = marker_prev_state[marker_prev_state["mutation"].isin(aq_present)] \
    .pivot_table(index="state", columns="mutation", values="prevalence", aggfunc="first")

aq_df = aq_df.reindex(index=state_order, columns=aq_present).fillna(0)

unique_state = list(aq_df.index)

nga = gpd.read_file("../assets/gadm41_NGA_1.shp").copy()

state_col = None
for c in ["NAME_1", "name_1", "NAME1", "state", "State"]:
    if c in nga.columns:
        state_col = c
        break

nga["state"] = nga[state_col].astype(str).str.strip()

state_fix_manifest = {
    "Federal Capital Territory": "Abuja Federal Capital Territory",
    "FCT": "Abuja Federal Capital Territory",
}
state_fix_map = {
    "Abuja Federal Capital Territory": "Abuja Federal Capital Territory",
}

aq_df = aq_df.rename(index=state_fix_manifest)
nga["state"] = nga["state"].replace(state_fix_map)

n_mut = len(aq_present)
ncols = 3
nrows = 2

#fig, axes = plt.subplots(nrows, ncols, figsize=(18, 18), dpi=600)
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10 * nrows / 2), dpi=600)
axes = np.array(axes).flatten()

letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("#f7f7f7")

for i, mut in enumerate(aq_present):
    ax = axes[i]

    tmp = aq_df[[mut]].reset_index().rename(columns={mut: "prevalence"})
    tmp["is_in_data"] = True

    map_df = nga.merge(tmp, on="state", how="left")
    map_df["plot_value"] = map_df["prevalence"]
    map_df.loc[map_df["plot_value"].isna(), "plot_value"] = np.nan
    map_df.loc[map_df["plot_value"] == 0, "plot_value"] = 0.001

    map_df.plot(
        column="plot_value",
        cmap=cmap,
        vmin=0,
        vmax=1,
        linewidth=0.8,
        edgecolor="black",
        ax=ax,
        legend=False,
        missing_kwds={"color": "#f7f7f7", "edgecolor": "black", "linewidth": 0.8}
    )

    ax.set_title(f"{letters[i]})  {mut}", loc="left", fontsize=16, fontweight="bold")
    ax.set_axis_off()

    for _, row in map_df.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue
        if pd.isna(row["prevalence"]):
            continue

        pt = row.geometry.representative_point()

        if float(row["prevalence"]) == 0:
            label_text = f"{row['state']}\n0.00"
            bbox_fc = "#fff2f0"
            txt_color = "#b2182b"
        else:
            label_text = f"{row['state']}\n{row['prevalence']:.2f}"
            bbox_fc = "white"
            txt_color = "black"

        ax.text(
            pt.x,
            pt.y,
            label_text,
            ha="center",
            va="center",
            fontsize=6.0,
            fontweight="bold",
            color=txt_color,
            bbox=dict(
                facecolor=bbox_fc,
                alpha=0.82,
                boxstyle="round,pad=0.16",
                linewidth=0.35,
                edgecolor="black"
            ),
            zorder=5
        )

for j in range(len(aq_present), len(axes)):
    axes[j].axis("off")

cax = fig.add_axes([0.92, 0.23, 0.015, 0.54])
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Prevalence", fontweight="bold", fontsize=13)
for t in cbar.ax.get_yticklabels():
    t.set_fontweight("bold")

fig.suptitle("AQ marker prevalence by state", fontsize=22, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 0.96])

save_figure(fig, "figure_step2_aq_marker_maps_multi_panel")
plt.show()

In [ ]:

# Figure 10. Marker denominator heatmap by state
marker_den = marker_prev_state.pivot_table(index='state', columns='mutation', values='denominator', aggfunc='first')
marker_den = marker_den.reindex(index=state_order)
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(marker_den.values, aspect='auto', cmap='cividis')
ax.set_xticks(range(marker_den.shape[1]))
ax.set_xticklabels(marker_den.columns, rotation=45, ha='right', fontweight='bold')
ax.set_yticks(range(marker_den.shape[0]))
ax.set_yticklabels(marker_den.index, fontweight='bold')
ax.set_title('Callable sample denominators for marker prevalence', fontweight='bold')
ax.set_xlabel('Mutation', fontweight='bold')
ax.set_ylabel('State', fontweight='bold')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Callable samples', fontweight='bold')
for t in cbar.ax.get_yticklabels():
    t.set_fontweight('bold')
#save_figure(fig, 'figure_step2_10_marker_denominator_heatmap_by_state')
#plt.show()


In [ ]:

# Figure 11. State-level marker callability versus prevalence
plot_df = marker_prev_state.merge(marker_callability[['state', 'mutation', 'callable_fraction']], on=['state', 'mutation'], how='left')
fig, ax = plt.subplots(figsize=(8, 6))
colors = plot_df['drug'].map({'SP': '#d7301f', 'AQ': '#225ea8'}).fillna('#636363')
ax.scatter(plot_df['callable_fraction'], plot_df['prevalence'], s=60, c=colors, edgecolor='black', alpha=0.8)
ax.set_xlabel('Callable fraction', fontweight='bold')
ax.set_ylabel('Prevalence', fontweight='bold')
ax.set_title('Marker prevalence as a function of fragment callability', fontweight='bold')
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#d7301f', markeredgecolor='black', markersize=9, label='SP'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#225ea8', markeredgecolor='black', markersize=9, label='AQ')
]
leg = ax.legend(handles=legend_elems, frameon=False, title='Drug class')
for t in leg.get_texts():
    t.set_fontweight('bold')
leg.get_title().set_fontweight('bold')
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontweight('bold')
#save_figure(fig, 'figure_step2_11_callability_vs_prevalence_scatter')
#plt.show()


## 4. Haplotype burden and composition

In [ ]:
hap_annotation = {
    "dhfr_triple": "\n(N51I + C59R + S108N)",
    "sp_quadruple": "\n(A437G + N51I + C59R\n + S108N)",
    "sp_quintuple": "\n(A437G + K540E + N51I\n + C59R + S108N)",
    "sp_sextuple": "\n(A437G + K540E + A581G\n + N51I + C59R + S108N)",
    "sp_sextuple_dhfr164": "\n(A437G + K540E + N51I\n + C59R + S108N + I164L)",
    "mdr1_86Y": "\n(N86Y)",
    "mdr1_184F": "\n(Y184F)",
    "mdr1_double": "\n(N86Y + Y184F)",
    "pfcrt_76T": "\n(K76T)",
    "pfcrt_triple": "\n(M74I + N75E + K76T)",
    "aq_resistant_profile": "\n(N86Y + Y184F + K76T)",
}

plot_df = hap_prev_overall.copy().sort_values(
    ["drug", "prevalence", "haplotype"],
    ascending=[True, False, True]
)

fig, ax = plt.subplots(figsize=(13, 7))

colors = ["#b2182b" if d == "SP" else "#2166ac" for d in plot_df["drug"]]
bars = ax.bar(
    range(len(plot_df)),
    plot_df["prevalence"],
    color=colors,
    edgecolor="black",
    linewidth=1.2
)

for i, row in plot_df.reset_index(drop=True).iterrows():
    ax.text(
        i,
        row["prevalence"] + 0.02,
        f"{int(row['numerator'])}/{int(row['denominator'])}",
        ha="center",
        va="bottom",
        rotation=90,
        fontsize=10,
        fontweight="bold"
    )

ax.set_ylim(0, 1.12)
ax.set_ylabel("Prevalence", fontsize=16, fontweight="bold")
ax.set_xlabel("\nHaplotype", fontsize=16, fontweight="bold")
ax.set_title("Overall prevalence of SP and AQ haplotypes", fontsize=22, fontweight="bold")

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(plot_df["haplotype"], rotation=45, ha="right")

for i, tick in enumerate(ax.get_xticklabels()):
    tick.set_fontweight("bold")
    tick.set_fontsize(11)

    annot = hap_annotation.get(plot_df.iloc[i]["haplotype"], "")
    if annot:
        ax.text(
            tick.get_position()[0],
            -0.08,
            annot,
            transform=ax.get_xaxis_transform(),
            ha="right",
            va="top",
            fontsize=8,
            fontweight="normal",
            rotation=45
        )

for lab in ax.get_yticklabels():
    lab.set_fontweight("bold")
    lab.set_fontsize(12)

legend_elems = [
    Line2D([0], [0], color="#b2182b", lw=10, label="SP haplotypes"),
    Line2D([0], [0], color="#2166ac", lw=10, label="AQ haplotypes"),
]
leg = ax.legend(
    handles=legend_elems,
    frameon=False,
    loc="upper right",
    fontsize=13
)
for t in leg.get_texts():
    t.set_fontweight("bold")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_linewidth(1.2)
ax.spines["bottom"].set_linewidth(1.2)

fig.tight_layout()
save_figure(fig, "figure_step2_12_overall_haplotype_prevalence_annotated")
plt.show()

In [ ]:
sp_hap_annotation = {
    "dhfr_triple": "Sulfadoxine–pyrimethamine\n(N51I + C59R + S108N)",
    "sp_quadruple": "Sulfadoxine–pyrimethamine\n(A437G + N51I + C59R + S108N)",
    "sp_quintuple": "Sulfadoxine–pyrimethamine\n(A437G + K540E + N51I + C59R + S108N)",
    "sp_sextuple": "Sulfadoxine–pyrimethamine\n(A437G + K540E + A581G + N51I + C59R + S108N)",
    "sp_sextuple_dhfr164": "Sulfadoxine–pyrimethamine\n(A437G + K540E + N51I + C59R + S108N + I164L)",
}

sp_hap_df = hap_prev_state[hap_prev_state["haplotype"].isin(sp_haps)] \
    .pivot_table(index="state", columns="haplotype", values="prevalence", aggfunc="first")

sp_hap_df = sp_hap_df.reindex(index=state_order, columns=sp_haps).fillna(0)

nga = gpd.read_file("../assets/gadm41_NGA_1.shp").copy()

state_col = None
for c in ["NAME_1", "name_1", "NAME1", "state", "State"]:
    if c in nga.columns:
        state_col = c
        break

nga["state"] = nga[state_col].astype(str).str.strip()

state_fix_manifest = {
    "Federal Capital Territory": "Abuja Federal Capital Territory",
    "FCT": "Abuja Federal Capital Territory",
}
state_fix_map = {
    "Abuja Federal Capital Territory": "Abuja Federal Capital Territory",
}

sp_hap_df = sp_hap_df.rename(index=state_fix_manifest)
nga["state"] = nga["state"].replace(state_fix_map)

valid_states = list(sp_hap_df.index)

n_hap = len(sp_haps)
ncols = 3
nrows = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10 * nrows / 2), dpi=600)
axes = np.array(axes).flatten()

letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("#f7f7f7")

for i, hap in enumerate(sp_haps):
    ax = axes[i]

    tmp = sp_hap_df[[hap]].reset_index().rename(columns={hap: "prevalence"})
    map_df = nga.merge(tmp, on="state", how="left")
    map_df["in_dataset"] = map_df["state"].isin(valid_states)
    map_df["prevalence"] = map_df["prevalence"].fillna(0)

    bg_df = map_df[~map_df["in_dataset"]].copy()
    fg_df = map_df[map_df["in_dataset"]].copy()
    fg_df["plot_value"] = fg_df["prevalence"].copy()
    fg_df.loc[fg_df["plot_value"] == 0, "plot_value"] = 0.001

    if len(bg_df) > 0:
        bg_df.plot(
            color="#f7f7f7",
            edgecolor="black",
            linewidth=0.8,
            ax=ax
        )

    fg_df.plot(
        column="plot_value",
        cmap=cmap,
        vmin=0,
        vmax=1,
        linewidth=0.8,
        edgecolor="black",
        ax=ax,
        legend=False
    )

    ax.set_title(f"{letters[i]})  {hap}", loc="left", fontsize=16, fontweight="bold")

    ann = sp_hap_annotation.get(hap, "")
    if ann:
        title_lines = ann.split("\n")
        ax.text(
            0.6,
            0.1,
            title_lines[0],
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )
        if len(title_lines) > 1:
            ax.text(
                0.6,
                0.08,
                title_lines[1],
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
                fontweight="normal"
            )

    ax.set_axis_off()

    for _, row in fg_df.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        pt = row.geometry.representative_point()

        if float(row["prevalence"]) == 0:
            label_text = f"{row['state']}\n0.00"
            bbox_fc = "#fff2f0"
            txt_color = "#b2182b"
        else:
            label_text = f"{row['state']}\n{row['prevalence']:.2f}"
            bbox_fc = "white"
            txt_color = "black"

        ax.text(
            pt.x,
            pt.y,
            label_text,
            ha="center",
            va="center",
            fontsize=5.8,
            fontweight="bold",
            color=txt_color,
            bbox=dict(
                facecolor=bbox_fc,
                alpha=0.82,
                boxstyle="round,pad=0.16",
                linewidth=0.35,
                edgecolor="black"
            ),
            zorder=5
        )

axes[len(sp_haps)].axis("off")

cax = fig.add_axes([0.92, 0.23, 0.015, 0.54])
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Prevalence", fontweight="bold", fontsize=13)
for t in cbar.ax.get_yticklabels():
    t.set_fontweight("bold")

fig.suptitle("SP haplotype prevalence by state", fontsize=22, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 0.96])

save_figure(fig, "figure_step2_sp_haplotype_maps_multi_panel")
plt.show()

In [ ]:
aq_hap_annotation = {
    "mdr1_184F": "Amodiaquine\n(Y184F)",
    "pfcrt_76T": "Amodiaquine\n(K76T)",
    "mdr1_86Y": "Amodiaquine\n(N86Y)",
    "mdr1_double": "Amodiaquine\n(N86Y + Y184F)",
    "aq_resistant_profile": "Amodiaquine\n(N86Y + Y184F + K76T)",
    "pfcrt_triple": "Amodiaquine\n(M74I + N75E + K76T)",
}

aq_hap_df = hap_prev_state[hap_prev_state["haplotype"].isin(aq_haps)] \
    .pivot_table(index="state", columns="haplotype", values="prevalence", aggfunc="first")

aq_hap_df = aq_hap_df.reindex(index=state_order, columns=aq_haps).fillna(0)

nga = gpd.read_file("../assets/gadm41_NGA_1.shp").copy()

state_col = None
for c in ["NAME_1", "name_1", "NAME1", "state", "State"]:
    if c in nga.columns:
        state_col = c
        break

nga["state"] = nga[state_col].astype(str).str.strip()

state_fix_manifest = {
    "Federal Capital Territory": "Abuja Federal Capital Territory",
    "FCT": "Abuja Federal Capital Territory",
}
state_fix_map = {
    "Abuja Federal Capital Territory": "Abuja Federal Capital Territory",
}

aq_hap_df = aq_hap_df.rename(index=state_fix_manifest)
nga["state"] = nga["state"].replace(state_fix_map)

valid_states = list(aq_hap_df.index)

n_hap = len(aq_haps)
ncols = 3
nrows = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 10 * nrows / 2), dpi=600)
axes = np.array(axes).flatten()

letters = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")
cmap = plt.cm.YlOrRd.copy()
cmap.set_bad("#f7f7f7")

for i, hap in enumerate(aq_haps):
    ax = axes[i]

    tmp = aq_hap_df[[hap]].reset_index().rename(columns={hap: "prevalence"})
    map_df = nga.merge(tmp, on="state", how="left")
    map_df["in_dataset"] = map_df["state"].isin(valid_states)
    map_df["prevalence"] = map_df["prevalence"].fillna(0)

    bg_df = map_df[~map_df["in_dataset"]].copy()
    fg_df = map_df[map_df["in_dataset"]].copy()
    fg_df["plot_value"] = fg_df["prevalence"].copy()
    fg_df.loc[fg_df["plot_value"] == 0, "plot_value"] = 0.001

    if len(bg_df) > 0:
        bg_df.plot(
            color="#f7f7f7",
            edgecolor="black",
            linewidth=0.8,
            ax=ax
        )

    fg_df.plot(
        column="plot_value",
        cmap=cmap,
        vmin=0,
        vmax=1,
        linewidth=0.8,
        edgecolor="black",
        ax=ax,
        legend=False
    )

    ax.set_title(f"{letters[i]})  {hap}", loc="left", fontsize=16, fontweight="bold")

    ann = aq_hap_annotation.get(hap, "")
    if ann:
        title_lines = ann.split("\n")
        ax.text(
            0.6,
            0.1,
            title_lines[0],
            transform=ax.transAxes,
            ha="left",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )
        if len(title_lines) > 1:
            ax.text(
                0.6,
                0.08,
                title_lines[1],
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
                fontweight="normal"
            )

    ax.set_axis_off()

    for _, row in fg_df.iterrows():
        if row.geometry is None or row.geometry.is_empty:
            continue

        pt = row.geometry.representative_point()

        if float(row["prevalence"]) == 0:
            label_text = f"{row['state']}\n0.00"
            bbox_fc = "#fff2f0"
            txt_color = "#b2182b"
        else:
            label_text = f"{row['state']}\n{row['prevalence']:.2f}"
            bbox_fc = "white"
            txt_color = "black"

        ax.text(
            pt.x,
            pt.y,
            label_text,
            ha="center",
            va="center",
            fontsize=5.8,
            fontweight="bold",
            color=txt_color,
            bbox=dict(
                facecolor=bbox_fc,
                alpha=0.82,
                boxstyle="round,pad=0.16",
                linewidth=0.35,
                edgecolor="black"
            ),
            zorder=5
        )

for j in range(len(aq_haps), len(axes)):
    ax = axes[j]
    bg_df = nga[~nga["state"].isin(valid_states)].copy()
    fg_df = nga[nga["state"].isin(valid_states)].copy()

    if len(bg_df) > 0:
        bg_df.plot(
            color="#f7f7f7",
            edgecolor="black",
            linewidth=0.8,
            ax=ax
        )

    if len(fg_df) > 0:
        fg_df.plot(
            color="#fff7bc",
            edgecolor="black",
            linewidth=0.8,
            ax=ax
        )

    ax.set_axis_off()

cax = fig.add_axes([0.92, 0.23, 0.015, 0.54])
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Prevalence", fontweight="bold", fontsize=13)
for t in cbar.ax.get_yticklabels():
    t.set_fontweight("bold")

fig.suptitle("AQ haplotype prevalence by state", fontsize=22, fontweight="bold")
fig.tight_layout(rect=[0, 0, 0.9, 0.96])

save_figure(fig, "figure_step2_aq_haplotype_maps_multi_panel")
plt.show()

In [ ]:
marker_only = marker_matrix.copy()
marker_cols = [c for c in marker_only.columns if c in sp_markers + aq_markers]
marker_only['mutation_burden'] = marker_only[marker_cols].fillna(0).sum(axis=1)

plot_df = marker_only[['state', 'mutation_burden']].dropna().copy()
state_sorted = plot_df.groupby('state')['mutation_burden'].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(12, 6))

data = [plot_df.loc[plot_df['state'] == st, 'mutation_burden'].values for st in state_sorted]

box = ax.boxplot(
    data,
    patch_artist=True,
    labels=state_sorted,
    boxprops=dict(facecolor='#fdae6b', color='black'),
    medianprops=dict(color='black', linewidth=0),  # hide default median line
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)

for i, d in enumerate(data, start=1):
    median_val = np.median(d)
    ax.plot(i, median_val, marker='*', color='black', markersize=10)

ax.set_ylabel('Number of resistance markers present per sample', fontweight='bold')
ax.set_xlabel('State', fontweight='bold')
ax.set_title('Sample-level resistance marker burden by state', fontweight='bold')

ax.set_xticklabels(state_sorted, rotation=60, ha='right', fontweight='bold')
for label in ax.get_yticklabels():
    label.set_fontweight('bold')

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor='#fdae6b', edgecolor='black', label='Interquartile range (IQR)'),
    Line2D([0], [0], marker='*', color='black', linestyle='None', markersize=10, label='Median'),
    Line2D([0], [0], color='black', lw=1, linestyle='-', label='Whiskers (min–max)'),
    Line2D([0], [0], marker='o', color='black', linestyle='None', label='Outliers')
]

leg = ax.legend(
    handles=legend_elements,
    frameon=True,
    loc='upper left',
    bbox_to_anchor=(1, 1),
    fontsize=10
)
for t in leg.get_texts():
    t.set_fontweight('bold')
leg.get_frame().set_edgecolor('black')
leg.get_frame().set_linewidth(1.0)

save_figure(fig, 'figure_step2_15_mutation_burden_boxplot_by_state')
plt.show()

In [ ]:
hap_only = hap_matrix.copy()
hap_cols = [c for c in hap_only.columns if c in sp_haps + aq_haps]
hap_only["haplotype_burden"] = hap_only[hap_cols].fillna(0).sum(axis=1)

plot_df = hap_only[["state", "haplotype_burden"]].dropna().copy()
state_sorted = plot_df.groupby("state")["haplotype_burden"].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(12, 6))

data = [plot_df.loc[plot_df["state"] == st, "haplotype_burden"].values for st in state_sorted]

box = ax.boxplot(
    data,
    patch_artist=True,
    labels=state_sorted,
    boxprops=dict(facecolor="#9ecae1", color="black"),
    medianprops=dict(color="black", linewidth=0),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    flierprops=dict(marker="o", markerfacecolor="black", markeredgecolor="black", markersize=6)
)

for i, d in enumerate(data, start=1):
    median_val = np.median(d)
    ax.plot(i, median_val, marker="*", color="black", markersize=10)

ax.set_ylabel("Number of haplotypes present per sample", fontweight="bold")
ax.set_xlabel("State", fontweight="bold")
ax.set_title("Sample-level haplotype burden by state", fontweight="bold")

ax.set_xticklabels(state_sorted, rotation=60, ha="right", fontweight="bold")
for label in ax.get_yticklabels():
    label.set_fontweight("bold")

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="#9ecae1", edgecolor="black", label="Interquartile range (IQR)"),
    Line2D([0], [0], marker="*", color="black", linestyle="None", markersize=14, label="Median"),
    Line2D([0], [0], color="black", lw=1.5, linestyle="-", label="Whiskers (min–max)"),
    Line2D([0], [0], marker="o", color="black", linestyle="None", markersize=8, label="Outliers")
]

leg = ax.legend(
    handles=legend_elements,
    frameon=True,
    loc='upper left',
    bbox_to_anchor=(1, 1),
    fontsize=10
)
for t in leg.get_texts():
    t.set_fontweight('bold')
leg.get_frame().set_edgecolor('black')
leg.get_frame().set_linewidth(1.0)

save_figure(fig, "figure_step2_16_haplotype_burden_boxplot_by_state")
plt.show()

## Link between resistance markers and haplotypes

In [ ]:
step3_mut = pd.read_csv(Base_step3 / "step3_mutation_matrix_global_high_quality_samples.tsv", sep="\t")
step3_hap = pd.read_csv(Base_step3 / "step3_haplotype_matrix_global_high_quality_samples.tsv", sep="\t")

sample_meta_cols = {"sample_id", "sample", "state", "year", "age_group"}
mut_cols = [c for c in step3_mut.columns if c not in sample_meta_cols]
hap_cols = [c for c in step3_hap.columns if c not in sample_meta_cols]

mut_df = step3_mut[mut_cols].fillna(0).astype(int)
hap_df = step3_hap[hap_cols].fillna(0).astype(int)

sp_mut_set = set(sp_markers)
aq_mut_set = set(aq_markers)
sp_hap_set = set(sp_haps)
aq_hap_set = set(aq_haps)

hap_annotation = {
    "dhfr_triple": "(N51I + C59R + S108N)",
    "sp_quadruple": "(A437G + N51I + C59R + S108N)",
    "sp_quintuple": "(A437G + K540E + N51I + C59R + S108N)",
    "sp_sextuple": "(A437G + K540E + A581G + N51I + C59R + S108N)",
    "sp_sextuple_dhfr164": "(A437G + K540E + N51I + C59R + S108N + I164L)",
    "mdr1_86Y": "(N86Y)",
    "mdr1_184F": "(Y184F)",
    "mdr1_double": "(N86Y + Y184F)",
    "pfcrt_76T": "(K76T)",
    "pfcrt_triple": "(M74I + N75E + K76T)",
    "aq_resistant_profile": "(N86Y + Y184F + K76T)",
}

def cooccurrence_matrix(df):
    arr = df.to_numpy(dtype=int)
    n = arr.shape[1]
    out = np.zeros((n, n), dtype=float)
    for i in range(n):
        xi = arr[:, i]
        xi_sum = xi.sum()
        for j in range(n):
            if i == j:
                out[i, j] = 1.0
            else:
                if xi_sum == 0:
                    out[i, j] = 0.0
                else:
                    out[i, j] = (xi & arr[:, j]).sum() / xi_sum
    return pd.DataFrame(out, index=df.columns, columns=df.columns)

mut_co = cooccurrence_matrix(mut_df)
hap_co = cooccurrence_matrix(hap_df)

mut_order = sorted(mut_co.index, key=lambda x: (0 if x in sp_mut_set else 1, x))
hap_order = sorted(hap_co.index, key=lambda x: (0 if x in sp_hap_set else 1, x))

mut_co = mut_co.loc[mut_order, mut_order].fillna(0)
hap_co = hap_co.loc[hap_order, hap_order].fillna(0)

mut_mask = np.triu(np.ones_like(mut_co, dtype=bool), k=1)
hap_mask = np.triu(np.ones_like(hap_co, dtype=bool), k=1)

fig, axes = plt.subplots(
    1, 2,
    figsize=(21, 9),
    dpi=600,
    gridspec_kw={"width_ratios": [1, 1], "wspace": 0.55}
)

ax1, ax2 = axes
cmap = plt.cm.YlOrRd.copy()

m1 = np.ma.masked_where(mut_mask, mut_co.values)
im1 = ax1.imshow(m1, cmap=cmap, vmin=0, vmax=1, aspect="equal")

ax1.set_xticks(range(len(mut_co.columns)))
ax1.set_yticks(range(len(mut_co.index)))
ax1.set_xticklabels(mut_co.columns, rotation=60, ha="right", fontsize=10, fontweight="bold")
ax1.set_yticklabels(mut_co.index, fontsize=10, fontweight="bold")
ax1.set_title("Pairwise co-occurrence between mutations", fontsize=18, fontweight="bold", pad=18)
ax1.set_xlabel("Mutation", fontsize=14, fontweight="bold")
ax1.set_ylabel("Mutation", fontsize=14, fontweight="bold")

for i in range(mut_co.shape[0]):
    for j in range(mut_co.shape[1]):
        if j > i:
            continue
        val = mut_co.iloc[i, j]
        txt_color = "white" if val >= 0.75 else "black"
        ax1.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7.2, fontweight="bold", color=txt_color)

for i, tick in enumerate(ax1.get_xticklabels()):
    tick.set_color("blue" if mut_co.columns[i] in sp_mut_set else "red")
for i, tick in enumerate(ax1.get_yticklabels()):
    tick.set_color("blue" if mut_co.index[i] in sp_mut_set else "red")

hap_display = []
for h in hap_co.columns:
    ann = hap_annotation.get(h, "")
    hap_display.append(f"{h}\n{ann}" if ann else h)

m2 = np.ma.masked_where(hap_mask, hap_co.values)
im2 = ax2.imshow(m2, cmap=cmap, vmin=0, vmax=1, aspect="equal")

ax2.set_xticks(range(len(hap_co.columns)))
ax2.set_yticks(range(len(hap_co.index)))
ax2.set_xticklabels(hap_display, rotation=60, ha="right", fontsize=8.5)
ax2.set_yticklabels(hap_display, fontsize=8.8)
ax2.set_title("Pairwise co-occurrence between haplotypes", fontsize=18, fontweight="bold", pad=18)
ax2.set_xlabel("Haplotype", fontsize=14, fontweight="bold")
ax2.set_ylabel("Haplotype", fontsize=14, fontweight="bold")

for i in range(hap_co.shape[0]):
    for j in range(hap_co.shape[1]):
        if j > i:
            continue
        val = hap_co.iloc[i, j]
        txt_color = "white" if val >= 0.75 else "black"
        ax2.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6.8, fontweight="bold", color=txt_color)

for i, tick in enumerate(ax2.get_xticklabels()):
    tick.set_fontweight("bold")
    tick.set_color("blue" if hap_co.columns[i] in sp_hap_set else "red")
for i, tick in enumerate(ax2.get_yticklabels()):
    tick.set_fontweight("bold")
    tick.set_color("blue" if hap_co.index[i] in sp_hap_set else "red")

for lbl in ax2.get_xticklabels():
    if "\n" in lbl.get_text():
        lbl.set_linespacing(1.12)

ax1.text(-0.1, 1.01, "A", transform=ax1.transAxes, fontsize=28, fontweight="bold")
ax2.text(-0.1, 1.01, "B", transform=ax2.transAxes, fontsize=28, fontweight="bold")

cbar = fig.colorbar(im2, ax=axes, fraction=0.025, pad=0.03)
cbar.set_label("Co-occurrence coefficient", fontsize=14, fontweight="bold")
cbar.ax.tick_params(labelsize=11)
for t in cbar.ax.get_yticklabels():
    t.set_fontweight("bold")

legend_elements = [
    Line2D([0], [0], color="blue", lw=4, label="SP (Sulfadoxine-Pyrimethamine)"),
    Line2D([0], [0], color="red", lw=4, label="AQ (Amodiaquine)"),
]

leg = fig.legend(
    handles=legend_elements,
    loc="upper left",
    bbox_to_anchor=(0.93, 0.93),
    ncol=1,
    frameon=True,
    fontsize=11,
    title="Label groups",
    borderaxespad=0.0
)
leg.get_title().set_fontweight("bold")
for t in leg.get_texts():
    t.set_fontweight("bold")

fig.suptitle(
    "Global pairwise co-occurrence structure of resistance mutations and haplotypes",
    fontsize=22,
    fontweight="bold",
    y=0.985
)

fig.tight_layout(rect=[0, 0, 0.9, 0.95])
save_figure(fig, "figure_step3_pairwise_cooccurrence_mutations_haplotypes")
plt.show()

In [ ]:
from itertools import combinations
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

mut_matrix = pd.read_csv(Base_step3 / "step3_mutation_matrix_global_high_quality_samples.tsv", sep="\t", index_col=0)
hap_matrix = pd.read_csv(Base_step3 / "step3_haplotype_matrix_global_high_quality_samples.tsv", sep="\t", index_col=0)

def compute_pairwise_stats(matrix):
    results = []
    cols = matrix.columns.tolist()
    for a, b in combinations(cols, 2):
        x = matrix[a]
        y = matrix[b]
        both = int(((x == 1) & (y == 1)).sum())
        only_a = int(((x == 1) & (y == 0)).sum())
        only_b = int(((x == 0) & (y == 1)).sum())
        none = int(((x == 0) & (y == 0)).sum())
        table = [[both, only_a], [only_b, none]]
        try:
            odds, p = fisher_exact(table)
        except:
            odds, p = np.nan, np.nan
        phi = ((both * none) - (only_a * only_b)) / math.sqrt((both + only_a)*(only_b + none)*(both + only_b)*(only_a + none)) if (both + only_a)*(only_b + none)*(both + only_b)*(only_a + none) > 0 else np.nan
        results.append([a, b, both, only_a, only_b, none, odds, p, phi])
    df = pd.DataFrame(results, columns=["feature_1","feature_2","both","only_1","only_2","none","odds_ratio","p_value","phi"])
    df["p_adj"] = multipletests(df["p_value"], method="fdr_bh")[1]
    return df

mut_stats = compute_pairwise_stats(mut_matrix)
hap_stats = compute_pairwise_stats(hap_matrix)

#print(mut_stats)
#print (hap_stats)

mut_stats.to_csv(Base_step3 / "step3_mutation_pairwise_stats.tsv", sep="\t", index=False)
hap_stats.to_csv(Base_step3 / "step3_haplotype_pairwise_stats.tsv", sep="\t", index=False)

mut_matrix = mut_matrix.select_dtypes(include=[np.number])
hap_matrix = hap_matrix.select_dtypes(include=[np.number])

mut_stats = compute_pairwise_stats(mut_matrix)
hap_stats = compute_pairwise_stats(hap_matrix)

mut_stats_clean = mut_stats.copy()
hap_stats_clean = hap_stats.copy()

df = mut_stats_clean.copy()
df["log10_p"] = -np.log10(df["p_adj"].replace(0, 1e-300))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18,10))

mut_plot = mut_stats_clean.copy()
mut_plot = mut_plot[
    (~mut_plot["feature_1"].astype(str).str.contains("year", case=False, na=False)) &
    (~mut_plot["feature_2"].astype(str).str.contains("year", case=False, na=False))
].copy()
mut_plot["log10_p"] = -np.log10(mut_plot["p_adj"].replace(0, 1e-300))

ax = axes[0,0]
ax.scatter(mut_plot["phi"], mut_plot["log10_p"], alpha=0.7)
ax.axhline(-np.log10(0.05), linestyle="--")
ax.axvline(0, linestyle="--")
for _, r in mut_plot.iterrows():
    if r["p_adj"] < 0.05 and abs(r["phi"]) > 0.3:
        ax.text(r["phi"], r["log10_p"], f"{r['feature_1']}-{r['feature_2']}", fontsize=8)
ax.set_xlabel("Phi", fontweight="bold")
ax.set_ylabel("-log10(FDR)", fontweight="bold")
ax.set_title("A) Mutation volcano", fontweight="bold")
for lab in ax.get_xticklabels() + ax.get_yticklabels():
    lab.set_fontweight("bold")

top_mut = mut_plot.sort_values(["p_adj", "phi"], ascending=[True, False]).head(15).copy()
top_mut["pair"] = top_mut["feature_1"] + " - " + top_mut["feature_2"]

ax = axes[0,1]
ax.barh(top_mut["pair"], top_mut["phi"], color="#2c7fb8", edgecolor="black")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("B) Top mutation associations", fontweight="bold")
ax.set_xlabel("Phi", fontweight="bold")
ax.invert_yaxis()
for lab in ax.get_xticklabels() + ax.get_yticklabels():
    lab.set_fontweight("bold")

sig_mut = mut_plot[mut_plot["p_adj"] < 0.05].copy()
mat_mut = pd.pivot_table(sig_mut, values="phi", index="feature_1", columns="feature_2")

ax = axes[0,2]
if mat_mut.shape[0] > 0 and mat_mut.shape[1] > 0:
    im1 = ax.imshow(mat_mut, aspect="auto")
    for i in range(mat_mut.shape[0]):
        for j in range(mat_mut.shape[1]):
            val = mat_mut.iloc[i, j]
            if not pd.isna(val):
                color = "white" if val < 0.5 else "black"
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10 , color=color ,fontweight="bold")
    ax.set_xticks(range(len(mat_mut.columns)))
    ax.set_xticklabels(mat_mut.columns, rotation=90, fontweight="bold")
    ax.set_yticks(range(len(mat_mut.index)))
    ax.set_yticklabels(mat_mut.index, fontweight="bold")
    cbar1 = fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.10)
    cbar1.set_label("Phi", fontweight="bold")
    for lab in cbar1.ax.get_yticklabels():
        lab.set_fontweight("bold")
else:
    ax.axis("off")
ax.set_title("C) Mutation heatmap", fontweight="bold")

hap_plot = hap_stats_clean.copy()
hap_plot = hap_plot[
    (~hap_plot["feature_1"].astype(str).str.contains("year", case=False, na=False)) &
    (~hap_plot["feature_2"].astype(str).str.contains("year", case=False, na=False))
].copy()
hap_plot["log10_p"] = -np.log10(hap_plot["p_adj"].replace(0, 1e-300))

ax = axes[1,0]
ax.scatter(hap_plot["phi"], hap_plot["log10_p"], alpha=0.7)
ax.axhline(-np.log10(0.05), linestyle="--")
ax.axvline(0, linestyle="--")
for _, r in hap_plot.iterrows():
    if r["p_adj"] < 0.05 and abs(r["phi"]) > 0.3:
        ax.text(r["phi"], r["log10_p"], f"{r['feature_1']}-{r['feature_2']}", fontsize=8)
ax.set_xlabel("Phi", fontweight="bold")
ax.set_ylabel("-log10(FDR)", fontweight="bold")
ax.set_title("D) Haplotype volcano", fontweight="bold")
for lab in ax.get_xticklabels() + ax.get_yticklabels():
    lab.set_fontweight("bold")

top_hap = hap_plot.sort_values(["p_adj", "phi"], ascending=[True, False]).head(15).copy()
top_hap["pair"] = top_hap["feature_1"] + " - " + top_hap["feature_2"]

ax = axes[1,1]
ax.barh(top_hap["pair"], top_hap["phi"], color="#2c7fb8", edgecolor="black")
ax.axvline(0, color="black", linewidth=1)
ax.set_title("E) Top haplotype associations", fontweight="bold")
ax.set_xlabel("Phi", fontweight="bold")
ax.invert_yaxis()
for lab in ax.get_xticklabels() + ax.get_yticklabels():
    lab.set_fontweight("bold")

sig_hap = hap_plot[hap_plot["p_adj"] < 0.05].copy()
mat_hap = pd.pivot_table(sig_hap, values="phi", index="feature_1", columns="feature_2")

ax = axes[1,2]
if mat_hap.shape[0] > 0 and mat_hap.shape[1] > 0:
    im2 = ax.imshow(mat_hap, aspect="auto")
    for i in range(mat_hap.shape[0]):
        for j in range(mat_hap.shape[1]):
            val = mat_hap.iloc[i, j]
            if not pd.isna(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=10 , color="#000000" ,fontweight="bold" )
    ax.set_xticks(range(len(mat_hap.columns)))
    ax.set_xticklabels(mat_hap.columns, rotation=90, fontweight="bold")
    ax.set_yticks(range(len(mat_hap.index)))
    ax.set_yticklabels(mat_hap.index, fontweight="bold")
    cbar2 = fig.colorbar(im2, ax=ax, fraction=0.046, pad=0.10)
    cbar2.set_label("Phi", fontweight="bold")
    for lab in cbar2.ax.get_yticklabels():
        lab.set_fontweight("bold")
else:
    ax.axis("off")
ax.set_title("F) Haplotype heatmap", fontweight="bold")

fig.suptitle("Pairwise association summary", fontsize=20, fontweight="bold", y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.96])
save_figure(fig, "Figure_step3_pairwise_association_panel")
plt.show()

In [ ]:
mut_matrix = pd.read_csv(Base_step3 / "step3_mutation_matrix_global_high_quality_samples.tsv", sep="\t", index_col=0)
hap_matrix = pd.read_csv(Base_step3 / "step3_haplotype_matrix_global_high_quality_samples.tsv", sep="\t", index_col=0)

sp_mut = ["N51I","C59R","S108N","A437G","K540E","A581G","A613S"]
aq_mut = ["N86Y","Y184F","D1246Y","K76T","M74I","N75E"]

sp_hap = ["dhfr_triple","sp_quadruple","sp_quintuple","sp_sextuple","sp_sextuple_dhfr164"]
aq_hap = ["mdr1_86Y","mdr1_184F","mdr1_double","aq_resistant_profile"]

profile = pd.DataFrame(index=mut_matrix.index)

profile["SP_mut_count"] = mut_matrix[ [c for c in sp_mut if c in mut_matrix.columns] ].sum(axis=1)
profile["AQ_mut_count"] = mut_matrix[ [c for c in aq_mut if c in mut_matrix.columns] ].sum(axis=1)

profile["SP_hap_present"] = hap_matrix[ [c for c in sp_hap if c in hap_matrix.columns] ].max(axis=1)
profile["AQ_hap_present"] = hap_matrix[ [c for c in aq_hap if c in hap_matrix.columns] ].max(axis=1)

def classify(row):
    if row["SP_mut_count"] >= 3 and row["AQ_mut_count"] >= 2:
        return "High_SP+AQ"
    if row["SP_mut_count"] >= 3:
        return "SP_dominant"
    if row["AQ_mut_count"] >= 2:
        return "AQ_dominant"
    if row["SP_mut_count"] > 0 and row["AQ_mut_count"] > 0:
        return "Mixed_low"
    return "Low_or_none"

profile["combined_profile"] = profile.apply(classify, axis=1)

summary = profile["combined_profile"].value_counts(normalize=True).reset_index()
summary.columns = ["profile","proportion"]

profile.to_csv(Base_step3 / "step3_sample_combined_SP_AQ_profiles.tsv", sep="\t")
summary.to_csv(Base_step3 / "step3_combined_profile_summary.tsv", sep="\t", index=False)

In [ ]:
profile = pd.read_csv(Base_step3 / "step3_sample_combined_SP_AQ_profiles.tsv", sep="\t", index_col=0)
summary = pd.read_csv(Base_step3 / "step3_combined_profile_summary.tsv", sep="\t")

profile_order = ["Low_or_none", "Mixed_low", "AQ_dominant", "SP_dominant", "High_SP+AQ"]
profile_colors = {
    "Low_or_none": "#d9d9d9",
    "Mixed_low": "#fee08b",
    "AQ_dominant": "#4575b4",
    "SP_dominant": "#d73027",
    "High_SP+AQ": "#7b3294",
}

summary["profile"] = pd.Categorical(summary["profile"], categories=profile_order, ordered=True)
summary = summary.sort_values("profile").copy()
summary["n_samples"] = summary["profile"].map(profile["combined_profile"].value_counts()).fillna(0).astype(int)

profile_plot = profile.copy()
profile_plot["combined_profile"] = pd.Categorical(profile_plot["combined_profile"], categories=profile_order, ordered=True)
profile_plot = profile_plot.sort_values(["combined_profile", "SP_mut_count", "AQ_mut_count"]).copy()

heat_df = profile_plot[["SP_mut_count", "AQ_mut_count", "SP_hap_present", "AQ_hap_present"]].copy()
heat_df.columns = ["SP mutation count", "AQ mutation count", "SP haplotype present", "AQ haplotype present"]

fig, axes = plt.subplots(
    1, 2,
    figsize=(18, 15),
    gridspec_kw={"width_ratios": [1, 1.3]}
)

ax1, ax2 = axes

# --- BARPLOT ---
bars = ax1.bar(
    summary["profile"].astype(str),
    summary["proportion"],
    color=[profile_colors[p] for p in summary["profile"].astype(str)],
    edgecolor="black",
    linewidth=1
)

for i, row in summary.reset_index(drop=True).iterrows():
    ax1.text(
        i,
        row["proportion"] + 0.01,
        f"{row['n_samples']}",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

ax1.set_ylim(0, summary["proportion"].max() * 1.2)
ax1.set_ylabel("Proportion", fontsize=13)
ax1.set_xlabel("")
ax1.set_title("A) Distribution of combined SP/AQ profiles", fontsize=16, pad=10)

ax1.set_xticklabels(summary["profile"].astype(str), rotation=30, ha="right", fontsize=11)
ax1.tick_params(axis='y', labelsize=11)

ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# --- HEATMAP ---
im = ax2.imshow(heat_df.T, aspect="auto", cmap="YlOrRd", vmin=0, vmax=heat_df.values.max())

change_points = []
prev = None
for i, val in enumerate(profile_plot["combined_profile"].astype(str).tolist()):
    if prev is None:
        prev = val
        continue
    if val != prev:
        change_points.append(i - 0.5)
        prev = val

for x in change_points:
    ax2.axvline(x, color="black", linewidth=1)

centers = []
labels = []
start = 0
vals = profile_plot["combined_profile"].astype(str).tolist()
for i in range(1, len(vals) + 1):
    if i == len(vals) or vals[i] != vals[start]:
        centers.append((start + i - 1) / 2)
        labels.append(vals[start])
        start = i

ax2_top = ax2.secondary_xaxis("top")
ax2_top.set_xticks(centers)
ax2_top.set_xticklabels(labels, rotation=30, ha="left", fontsize=10)

ax2.set_yticks(range(len(heat_df.index)))
ax2.set_yticklabels(heat_df.index, fontsize=11)
ax2.set_xticks([])

ax2.set_xlabel("Samples", fontsize=13)
ax2.set_title("B) Sample-level resistance pattern", fontsize=16, pad=10)

# annotations plus légères (moins agressives)
for i in range(heat_df.shape[0]):
    for j in range(heat_df.shape[1]):
        val = heat_df.iloc[i, j]
        if val > 0:
            ax2.text(
                j,
                i,
                f"{int(val)}",
                ha="center",
                va="center",
                fontsize=6,
                color="black"
            )

# colorbar plus lisible
cbar = fig.colorbar(im, ax=ax2, fraction=0.035, pad=0.02)
cbar.set_label("Burden / presence", fontsize=12)

# légende simplifiée
legend_handles = [
    Patch(facecolor=profile_colors[p], edgecolor="black", label=p)
    for p in profile_order if p in summary["profile"].astype(str).tolist()
]

ax1.legend(
    handles=legend_handles,
    title="Profile",
    fontsize=10,
    title_fontsize=11,
    frameon=False
)

fig.suptitle(
    "Combined SP/AQ resistance profiles",
    fontsize=18,
    y=0.98
)

fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 5. Supplementary display tables

In [ ]:

# Top-level summary tables for direct manuscript use
summary_tables = {
    'QC by state': qc_state.sort_values('state'),
    'Inclusion flow by state': inclusion.sort_values('state'),
    'Overall marker prevalence': marker_prev_overall.sort_values(['drug', 'mutation']),
    'Overall haplotype prevalence': hap_prev_overall.sort_values(['drug', 'haplotype']),
}
for title, df in summary_tables.items():
    display(Markdown(f'### {title}'))
    display(df)


## 6. Exploratory mutation calls

In [ ]:

# The current Step 2 output contains no exploratory observed mutations.
# The notebook handles this explicitly so that the analysis remains reproducible.
if expl_state.empty and expl_overall.empty:
    display(Markdown('**No exploratory observed mutation calls were detected in the current Step 2 outputs.**'))
else:
    display(Markdown('### Exploratory mutation prevalence by state'))
    display(expl_state)
    display(Markdown('### Exploratory mutation prevalence overall'))
    display(expl_overall)


In [ ]:

# Figure export summary
exported_png = sorted(FIG_DIR.glob('*.png'))
exported_pdf = sorted(FIG_DIR.glob('*.pdf'))
print('PNG figures:', len(exported_png))
print('PDF figures:', len(exported_pdf))
for p in exported_png:
    print(p.name)
